In [10]:
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..", "..")))

from src.Generate.Generate_By_Api import BaseLLM

llm = BaseLLM()

In [11]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
import json
from langchain_core.messages import messages_to_dict, messages_from_dict
from langchain_core.callbacks import UsageMetadataCallbackHandler
cb=UsageMetadataCallbackHandler()

In [12]:
agent=create_agent(

    model=llm.llm,
    system_prompt="You are a helpful assistant.",

    middleware=[
        SummarizationMiddleware(
            model=llm.llm,
            trigger=("tokens",4000),
            keep=("messages" , 20)
        )
    ],
    checkpointer=InMemorySaver()
    )
config = {"configurable": {"thread_id": "user_1"}}

In [13]:
message_1 = "Can you list 3 places in Paris to visit?"

message_2 = "Which place has a lake view?"

message_3 = "Can I visit in winter?"

In [14]:
output_1=agent.invoke(
    {
        "messages" : [
            {
                "role" : "user",
                "content" : message_1
            }
        ]
    },
    config={
        "configurable" : {"thread_id" : "user_1"},
        "callbacks" : [cb]
    }

)

print(output_1["messages"][-1].content) 
print("======================")
print(cb.usage_metadata)

Here are three iconic places to visit in Paris:

1. **The Eiffel Tower**: The most recognizable symbol of France, this iron lattice tower offers breathtaking panoramic views of the city from its observation decks. You can also enjoy a meal at one of the restaurants inside or watch the sparkling light show at night.

2. **The Louvre Museum**: The world’s largest art museum and a historic monument in Paris, the Louvre houses over 35,000 works of art, including the *Mona Lisa* and the *Venus de Milo*. It’s a must-visit for art lovers and history enthusiasts.

3. **Notre-Dame Cathedral**: A masterpiece of French Gothic architecture located on the Île de la Cité. While it is currently undergoing restoration after the 2019 fire, the exterior and surrounding area remain stunning, and you can still appreciate its historical significance and architectural beauty.

Let me know if you’d like recommendations for other types of attractions, such as parks, neighborhoods, or dining spots!
{'qwen/qwen

In [15]:

output_2=agent.invoke(
    {
        "messages": [
            {
            "role" : "user" , 
            "content": message_2
            }
        ]
    },
    config={
        "configurable": { "thread_id": "user_1" }, 
        "callbacks": [cb]
    }
)
print(output_2["messages"][-1].content) 
print("======================")
print(cb.usage_metadata)

None of the three places listed above (the Eiffel Tower, the Louvre Museum, or Notre-Dame Cathedral) are primarily known for having a **lake view**.

However, if you are looking for a place in Paris with a lake view, here are a few excellent alternatives:

1. **Parc des Buttes-Chaumont**: This park features a large artificial lake with a small island and a temple-like structure (Temple de la Sybille) that offers beautiful views over the water and the city.
2. **Parc de la Villette**: Home to the largest lake in Paris, it’s a great spot for relaxation, boating, and enjoying the water.
3. **Parc de Sceaux** (just outside central Paris in the suburbs): A beautiful park with a large lake, ideal for a day trip if you’re willing to travel slightly beyond the city center.

Would you like more details on any of these?
{'qwen/qwen3.8-27b': {'output_tokens': 413, 'input_tokens': 303, 'total_tokens': 716}}


In [16]:


output_3=agent.invoke(
    {
        "messages" : [
            {
                "role" : "user" , 
                "content" : message_3
            }
        ]
    },
    config={
        "configurable" : {"thread_id" : "user_1"},
        "callback" : [cb]
    }
)
print(output_3["messages"][-1].content) 
print("======================")
print(cb.usage_metadata)

Yes, you can absolutely visit these places in winter! In fact, winter is a wonderful time to see Paris, though it comes with its own unique charm and considerations.

Here’s what to expect:

### **The Original Three:**
1. **Eiffel Tower**: 
   - **Pros**: Fewer crowds than in summer, and the city lights at night are especially magical against the dark sky. The tower is illuminated with a sparkling light show every hour after sunset.
   - **Cons**: It can be windy and cold on the open observation decks. Book tickets in advance to avoid long lines.

2. **Louvre Museum**: 
   - **Pros**: Indoor, warm, and less crowded. You can spend hours exploring without worrying about the weather.
   - **Cons**: None really—it’s a great winter activity.

3. **Notre-Dame Cathedral**: 
   - **Status**: As of 2024, Notre-Dame is **still under restoration** and **closed to the public** for interior visits. However, you can still admire the exterior, which looks stunning in winter, especially when dusted wi

In [21]:
state = agent.get_state(config)
dicts = messages_to_dict(state.values["messages"])

with open("conversation-memory.json", "w", encoding="utf-8") as dest:
    dest.write(json.dumps(dicts, ensure_ascii=False))

print("saved to:", os.path.abspath("conversation-memory.json"))
print("messages saved:", len(dicts))

saved to: f:\Prompt Engineering using LangChain\src\Conversation\conversation-memory.json
messages saved: 6


In [22]:
# ========== 2) التحميل والاستكمال ==========
with open("conversation-memory.json", encoding="utf-8") as src:   # مسار نسبي بدل /content/
    saved_history = json.loads(src.read())

restored = messages_from_dict(saved_history)

new_config = {"configurable": {"thread_id": "user_1_restored"}}

message_3 = "Can I visit in winter?"

result = agent.invoke(
    {"messages": [*restored, {"role": "user", "content": message_3}]},
    new_config,
)
print(result["messages"][-1].content)

Yes, you can absolutely visit these places in winter! In fact, winter is a wonderful time to see Paris, though it comes with its own unique charm and considerations.

Here’s what to expect:

### **The Original Three:**
1. **Eiffel Tower**: 
   - **Pros**: Fewer crowds than in summer, and the city lights at night are especially magical against the dark sky. The tower is illuminated with a sparkling light show every hour after sunset.
   - **Cons**: It can be windy and cold on the open observation decks. Book tickets in advance to avoid long lines.

2. **Louvre Museum**: 
   - **Pros**: Indoor, warm, and less crowded. You can spend hours exploring without worrying about the weather.
   - **Cons**: None really—it’s a great winter activity.

3. **Notre-Dame Cathedral**: 
   - **Status**: As of 2024, Notre-Dame is **still under restoration** and **closed to the public** for interior visits. However, you can still admire the exterior, which looks stunning in winter, especially when dusted wi